In [8]:
import pandas as pd

rides = pd.read_parquet('../data/transformed/validated_rides_2026_05.parquet')
rides.head(10)

,pickup_datetime,pickup_location_id
0,2026-05-01 00:04:59,138
1,2026-05-01 00:37:05,138
2,2026-05-01 00:34:05,249
3,2026-05-01 00:55:07,232
4,2026-05-01 00:44:13,140
5,2026-05-01 00:24:35,255
6,2026-05-01 00:06:08,43
7,2026-05-01 00:49:20,164
8,2026-05-01 00:29:38,234
9,2026-05-01 00:39:07,234


In [9]:
rides['pickup_datetime'].dtype

dtype('<M8[us]')

In [10]:
rides['pickup_hour'] = rides['pickup_datetime'].dt.floor('h')
rides

,pickup_datetime,pickup_location_id,pickup_hour
0,2026-05-01 00:04:59,138,2026-05-01 00:00:00
1,2026-05-01 00:37:05,138,2026-05-01 00:00:00
2,2026-05-01 00:34:05,249,2026-05-01 00:00:00
3,2026-05-01 00:55:07,232,2026-05-01 00:00:00
4,2026-05-01 00:44:13,140,2026-05-01 00:00:00
...,...,...,...
4090831,2026-05-31 23:35:12,74,2026-05-31 23:00:00
4090832,2026-05-31 23:48:01,129,2026-05-31 23:00:00
4090833,2026-05-31 23:48:50,48,2026-05-31 23:00:00
4090834,2026-05-31 23:23:13,113,2026-05-31 23:00:00


In [11]:
agg_rides = rides.groupby(['pickup_hour', 'pickup_location_id']).size().reset_index()
agg_rides.rename(columns={0: 'rides'}, inplace=True)
agg_rides

,pickup_hour,pickup_location_id,rides
0,2026-05-01 00:00:00,2,1
1,2026-05-01 00:00:00,3,1
2,2026-05-01 00:00:00,4,37
3,2026-05-01 00:00:00,6,1
4,2026-05-01 00:00:00,7,2
...,...,...,...
120530,2026-05-31 23:00:00,261,10
120531,2026-05-31 23:00:00,262,13
120532,2026-05-31 23:00:00,263,30
120533,2026-05-31 23:00:00,264,5


In [12]:
from tqdm import tqdm

def add_missing_slots(agg_rides: pd.DataFrame) -> pd.DataFrame:
    
    location_ids = agg_rides['pickup_location_id'].unique()
    full_range = pd.date_range(
        agg_rides['pickup_hour'].min(), agg_rides['pickup_hour'].max(), freq='h'
    )
    output = pd.DataFrame()
    for location_id in tqdm(location_ids):
        
        # keep only rides for this location ids
        agg_rides_i = agg_rides.loc[agg_rides.pickup_location_id == location_id, ['pickup_hour', 'rides']]
        
        # quick way to add missing hours with 0 rides
        agg_rides_i.set_index('pickup_hour', inplace=True)
        agg_rides_i.index = pd.DatetimeIndex(agg_rides_i.index)
        agg_rides_i = agg_rides_i.reindex(full_range, fill_value=0)
        
        # add back location id column
        agg_rides_i['pickup_location_id'] = location_id
        output = pd.concat([output, agg_rides_i])
        
    output = output.reset_index().rename(columns={'index': 'pickup_hour'})
    return output

In [13]:
agg_rides_all_slots = add_missing_slots(agg_rides)
agg_rides_all_slots

100%|██████████| 259/259 [00:00<00:00, 1942.36it/s]


,pickup_hour,rides,pickup_location_id
0,2026-05-01 00:00:00,1,2
1,2026-05-01 01:00:00,0,2
2,2026-05-01 02:00:00,0,2
3,2026-05-01 03:00:00,0,2
4,2026-05-01 04:00:00,0,2
...,...,...,...
192691,2026-05-31 19:00:00,0,176
192692,2026-05-31 20:00:00,0,176
192693,2026-05-31 21:00:00,0,176
192694,2026-05-31 22:00:00,0,176


In [14]:
print(agg_rides.empty)
print(agg_rides['pickup_hour'].dtype)
print(agg_rides['pickup_hour'].isna().sum())
print(agg_rides['pickup_hour'].min(), agg_rides['pickup_hour'].max())

False
datetime64[us]
0
2026-05-01 00:00:00 2026-05-31 23:00:00


In [15]:
from typing import Optional, List
import plotly.express as px
import pandas as pd

def plot_rides(
    rides: pd.DataFrame,
    locations: Optional[List[int]] = None
) -> None:
    
    """
    Plot time-series data
    """

    rides_to_plot = rides[rides.pickup_location_id.isin(locations)] if locations else rides

    fig = px.line(
        rides_to_plot,
        x='pickup_hour',
        y='rides',
        color='pickup_location_id',
        title='Rides per hour by pickup location'
    )

    fig.show()


In [16]:
plot_rides(agg_rides_all_slots, locations=[43])

In [17]:
agg_rides_all_slots.to_parquet('../data/transformed/transformed_ts_2026_05.parquet')